# 09b — The curated representativeness block on the Alberta extent and the manifest v3.1 freeze (zero solves)

Mirror of the parent's `11b_efg_curation_freeze_v3` (study plan v0.17–v0.17.3; AB spec v0.5 D-AB11/D-AB12; methods_log M15).
Kernel `y2y-geo`, minutes. **The curation is inherited, not re-decided:** rule R0 (purpose relevance, map validity, the
grain floor, the F2.9 utility drop, the two duplicate-footprint merges) is a property of each class's GET map, decided on the
parent and applied to the Alberta block by the parent's 11b (`input_data/aligned_stack_ab/iucn_efg_v3/`, 27 → 15 classes →
13 features). This notebook (1) re-applies the extent-relative parts of R0 on the Alberta extent as a DISCLOSURE (rule A's
envelope threshold, rule B's grain floor, the boundary-proximity check R0 iii) and asserts the block folder matches the
inherited set; (2) measures the block card v3 on Alberta (rare-attainable count, the ≤1%-footprint companion, leverage);
(3) derives the **rarity-scaled (log-linear) targets from each class's footprint in the ALBERTA extent buffered by 250 km**
(100/500 km sensitivity; zonal counts on the GET archive maps — the parent's v0.17.3 rule applied to this extent, D-AB12),
with the parent's own window targets printed beside them; (4) freezes **`spec/manifest_v3.1.csv`** — the 12 design
formulations (s1x/s3x are diagnostic, non-voting, not re-solved: parent v0.15/v0.17), weights re-derived on the AB stack and
asserted EQUAL to the v1 freeze (EFGs sit outside the block accounting), the EFG targets added by feature name, `role`,
block provenance, code provenance (git SHA + the five pipeline modules clean vs HEAD — commit `config.py` first).

v1 (`spec/manifest.csv`, `runs/ab_l/`, `analysis/ab4/`) stays byte-identical: supersede, never delete.

In [1]:
# ---- bootstrap + the inherited curation, re-read on the Alberta extent (disclosure) ---------------------------------
import hashlib, importlib, json, pathlib, shutil, subprocess, sys
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import rasterio
import geopandas as gpd
from scipy import ndimage
from pyproj import Transformer
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
assert _cands, "config.py not found above the notebook"
ROOT = _cands[0]; sys.path.insert(0, str(ROOT))
import config, leverage_core as lc
for _m in (config, lc):
    importlib.reload(_m)
VP = config.ab_paths()
assert VP.version == "v3.1" and config.EFG_SUBDIR == VP.efg_subdir == "iucn_efg_v3", (VP.version, config.EFG_SUBDIR)
HERE = ROOT / "analyses" / "alberta_prioritization"; SPEC, DATA = HERE / "spec", HERE / "data"
REC = VP.records; REC.mkdir(exist_ok=True)
AB = config.AB_HANDOFF_DIR; SRC, DST = AB / "iucn_efg", AB / config.EFG_SUBDIR
PARENT = pd.read_csv(config.y2y_paths("v3").records / "efg_curation_v3.csv").set_index("code")     # the parent's R0 record
MERGES = {m: tuple(PARENT.index[PARENT.merged_into == m]) for m in PARENT.merged_into.dropna().unique() if m}
GRAIN_KM2, ENVELOPE_MIN_PCT, BOUNDARY_KM, BOUNDARY_FLAG = 330, 1.0, 10, 0.50      # the parent's constants (11b)
code_of = lambda p: p.stem.split(".web")[0]
files = {code_of(p): p for p in sorted(SRC.glob("*.tif"))}
assert set(files) <= set(PARENT.index), set(files) - set(PARENT.index)
pu = lc.pu_mask(AB); n_pu = int(pu.sum())
with rasterio.open(AB / "mask_protected_areas.tif") as src:
    locked = (src.read(1) == 1) & pu
dist_in = ndimage.distance_transform_edt(pu)          # km to the nearest non-PU cell = the Alberta study boundary
rows = []
for c, p in files.items():
    pr = PARENT.loc[c]
    e = np.nan_to_num(lc._read(p), nan=0.0) > 0; pres = e & pu; n = int(pres.sum())
    r = dict(code=c, name=pr["name"], method=pr.method, anthropogenic=bool(pr.anthropogenic), cells=n, pct_pu=100 * n / n_pu,
             pct_in_PAs=100 * float((pres & locked).sum() / max(n, 1)),
             rule_C_purpose=bool(pr.anthropogenic), rule_A_point=(pr.method == "point"),
             rule_A_envelope_small_AB=(pr.method == "envelope" and 100 * n / n_pu < ENVELOPE_MIN_PCT),
             rule_B_subgrain_AB=(pr.method != "direct" and n < GRAIN_KM2), utility_drop=bool(pr.utility_drop),
             merged_into=(pr.merged_into if isinstance(pr.merged_into, str) else ""),
             retained=bool(pr.retained))                                                    # INHERITED from the parent
    r["extent_rule_would_drop"] = bool(r["retained"] and (r["rule_A_envelope_small_AB"] or r["rule_B_subgrain_AB"]))
    r["boundary_share_10km"] = 100 * float(pres[dist_in <= BOUNDARY_KM].sum() / max(n, 1)) if (r["retained"] and r["pct_pu"] < 1.0) else np.nan
    r["clip_edge_flag"] = bool(r["boundary_share_10km"] > 100 * BOUNDARY_FLAG) if r["retained"] and r["pct_pu"] < 1.0 else False
    r["reason"] = pr.reason if isinstance(pr.reason, str) else ""
    rows.append(r)
CUR = pd.DataFrame(rows).sort_values("cells").reset_index(drop=True)
RETAINED = sorted(CUR[CUR.retained].code)
FEATURES = [c for c in RETAINED if not CUR.set_index("code").loc[c, "merged_into"]] + [m for m, pair in MERGES.items() if any(c in RETAINED for c in pair)]
CUR.to_csv(REC / "efg_curation_v3.csv", index=False)
print(f"Alberta: {len(CUR)} classes present -> {len(RETAINED)} retained (inherited R0) -> {len(FEATURES)} features | dropped: "
      f"C {int(CUR.rule_C_purpose.sum())}, A-point {int(CUR.rule_A_point.sum())}, utility {int(CUR.utility_drop.sum())}, "
      f"B-subgrain (parent) {int((~CUR.retained & ~CUR.rule_C_purpose & ~CUR.rule_A_point & ~CUR.utility_drop).sum())}")
print(CUR[["code", "name", "method", "cells", "pct_pu", "pct_in_PAs", "retained", "merged_into", "extent_rule_would_drop", "boundary_share_10km", "clip_edge_flag"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))
print(f"\nextent-relative re-application (disclosure, never a drop): {list(CUR[CUR.extent_rule_would_drop].code) or 'no retained class fails rule A/B on the Alberta extent'}")
print(f"R0 (iii) boundary-proximity flags on Alberta (> {100*BOUNDARY_FLAG:.0f}% of a small retained class within {BOUNDARY_KM} km of the study boundary): "
      f"{list(CUR[CUR.clip_edge_flag].code) or 'none'} -- reported; the window-derived targets (next cells) are the standing remedy (parent v0.17.3)")
# the block folder built by the parent's 11b must be exactly the inherited set (rebuild if absent, same procedure)
def build_block(src, dst):
    dst.mkdir(exist_ok=True); have = {code_of(p): p for p in sorted(src.glob("*.tif"))}
    merged_into = set(m for pair in MERGES.values() for m in pair); written = []
    for c in RETAINED:
        if c not in have or c in merged_into: continue
        out = dst / have[c].name
        if not out.exists(): shutil.copy2(have[c], out)
        written.append(out)
    for mname, pair in MERGES.items():
        present = [have[c] for c in pair if c in have]
        if not present: continue
        out = dst / f"{mname}.web.merged_v3.tif"
        if not out.exists():
            with rasterio.open(present[0]) as s0: prof = s0.profile; a = s0.read(1)
            for q in present[1:]:
                with rasterio.open(q) as sq: b = sq.read(1)
                nd = prof.get("nodata", 255)
                a = np.where((a == nd) & (b == nd), nd, np.maximum(np.where(a == nd, 0, a), np.where(b == nd, 0, b))).astype(a.dtype)
            with rasterio.open(out, "w", **prof) as d: d.write(a, 1)
        written.append(out)
    return sorted(written)
BUILT = build_block(SRC, DST)
importlib.reload(lc)
got = sorted(p.stem for p in lc.efg_paths(AB)); exp = sorted(p.stem for p in BUILT)
assert got == exp and len(got) == len(FEATURES), (len(got), len(FEATURES), set(got) ^ set(exp))
print(f"\nblock folder {DST.relative_to(ROOT)}: {len(got)} features = the inherited curation (asserted)")


Alberta: 27 classes present -> 15 retained (inherited R0) -> 13 features | dropped: C 10, A-point 1, utility 1, B-subgrain (parent) 0
 code                                                    name   method  cells  pct_pu  pct_in_PAs  retained merged_into  extent_rule_would_drop  boundary_share_10km  clip_edge_flag
 T6.1           Ice sheets, glaciers and perennial snowfields   direct    247    0.29       97.98      True                               False                99.60            True
 F3.5                              Canals, ditches and drains envelope    254    0.30       67.72     False                               False                  NaN           False
SF2.2                           Flooded mines and other voids envelope    341    0.40        0.00     False                               False                  NaN           False
F2.10                                        Subglacial lakes    point    409    0.48        0.00     False                               Fals

In [2]:
# ---- block card v3 on the Alberta extent (mirror of the parent's 11b cell 3; audit convention = 30% of extent, M2.6) ----
with rasterio.open(AB / "cost_uniform.tif") as src:
    tr = src.transform
rr, cc = np.where(pu)
lon, lat = Transformer.from_crs(config.TARGET_CRS, "EPSG:4326", always_xy=True).transform(tr.c + (cc + 0.5) * tr.a, tr.f + (rr + 0.5) * tr.e)
LAT = lat.astype(np.float32); south = LAT < 53.0
rows = []
for p in lc.efg_paths(AB):
    v = np.nan_to_num(lc._read(p)[pu], nan=0.0); e = v > 0; n = int(e.sum())
    cmin, cmax, lev = lc.leverage_of(v)
    rows.append(dict(feature=p.stem, code=code_of(p), cells=n, pct_pu=100 * n / n_pu, pct_in_PAs=100 * float(e[locked[pu]].sum() / max(n, 1)),
                     banked_capture=float(v[locked[pu]].sum() / max(v.sum(), 1e-12)),
                     cap_max=float(cmax), leverage=float(lev), rare_attainable=bool(cmax >= config.AUDIT["rare_cap"]),
                     le1pct_footprint=bool(n <= 0.01 * n_pu), south_share=float(e[south].sum() / max(n, 1)), mean_lat=float(LAT[e].mean()),
                     target_on_extent=config.efg_target(n)))
CARD = pd.DataFrame(rows).sort_values("cells").reset_index(drop=True)
CARD.to_csv(REC / "efg_block_card_v3.csv", index=False)
n_rare, n_le1 = int(CARD.rare_attainable.sum()), int(CARD.le1pct_footprint.sum())
print(f"Alberta block card v3 ({len(CARD)} features): rare-attainable {n_rare}/{len(CARD)} (v1: 20/27, R2.3) | <=1%-footprint companion {n_le1} | "
      f"entirely inside PAs (banked >= 0.999): {int((CARD.banked_capture >= 0.999).sum())} | median banked {CARD.banked_capture.median():.3f}")
print(CARD[["code", "cells", "pct_pu", "pct_in_PAs", "banked_capture", "cap_max", "leverage", "rare_attainable", "le1pct_footprint", "mean_lat", "target_on_extent"]]
      .to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("\nNOTE 'target_on_extent' is the parent's superseded v3 rule (on-extent footprint) shown for the record; the frozen targets come from the buffered window (next cell)")


Alberta block card v3 (13 features): rare-attainable 9/13 (v1: 20/27, R2.3) | <=1%-footprint companion 1 | entirely inside PAs (banked >= 0.999): 0 | median banked 0.328
       code  cells  pct_pu  pct_in_PAs  banked_capture  cap_max  leverage  rare_attainable  le1pct_footprint  mean_lat  target_on_extent
       T6.1    247   0.290      97.976           0.980    1.000     1.000             True              True    52.184             1.000
       T4.4   1371   1.610       2.553           0.026    1.000     1.000             True             False    50.433             0.949
       T5.1   2391   2.809       0.125           0.001    1.000     1.000             True             False    49.726             0.858
TF1.6_TF1.7   3919   4.603       0.000           0.000    1.000     1.000             True             False    54.728             0.777
       T2.2   5290   6.214       0.662           0.005    1.000     1.000             True             False    53.615             0.728
       T

In [3]:
# ---- D-AB12: targets from each class's footprint in the ALBERTA extent buffered by 250 km (100 / 500 km sensitivity) -----
# The parent's v0.17.3 rule applied to this extent: rarity measured inside the study polygon made range edges of classes
# abundant just outside the line look rare, so the target is derived from a zonal count on the GET archive maps within the
# extent buffered by EFG_TARGET_WINDOW_KM. The parent's own +250 km targets for the same features are printed beside
# Alberta's (the parent's window is the Y2Y extent; a regional plan's rarity is judged in its own region).
import rasterio.features as rfeatures
import rasterio.windows
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
SRC_GET = config.DATASETS["iucn_efg"]["path"]
W_MAIN, W_SENS = config.EFG_TARGET_WINDOW_KM, config.EFG_TARGET_WINDOW_SENS_KM
WINDOWS = sorted({W_MAIN, *W_SENS})
ext = gpd.read_file(DATA / "ab_extent_v1.gpkg").to_crs(config.TARGET_CRS).union_all()      # the AB study extent (M2.1)
polys = {w: ext.buffer(w * 1000.0) for w in WINDOWS}
big = polys[max(WINDOWS)]
x0, y0, x1, y1 = big.bounds; res = config.TARGET_RES_M
x0, y1 = np.floor(x0 / res) * res, np.ceil(y1 / res) * res
W_, H_ = int(np.ceil((x1 - x0) / res)), int(np.ceil((y1 - y0) / res))
tr_win = from_origin(x0, y1, res, res)
win_masks = {w: rfeatures.rasterize([(polys[w], 1)], out_shape=(H_, W_), transform=tr_win, fill=0, dtype="uint8").astype(bool) for w in WINDOWS}
ext_mask = rfeatures.rasterize([(ext, 1)], out_shape=(H_, W_), transform=tr_win, fill=0, dtype="uint8").astype(bool)
_tf = Transformer.from_crs(config.TARGET_CRS, "EPSG:4326", always_xy=True)
gx = np.linspace(x0, x0 + W_ * res, 60); gy = np.linspace(y1 - H_ * res, y1, 60)
lonw, latw = _tf.transform(*np.meshgrid(gx, gy))
bb = (lonw.min() - 1, latw.min() - 1, lonw.max() + 1, latw.max() + 1)
def window_presence(c):
    with rasterio.open(SRC_GET / files[c].name) as src:
        rw = rasterio.windows.from_bounds(*bb, src.transform).round_offsets().round_lengths()
        arr = src.read(1, window=rw); src_tr = src.window_transform(rw); src_crs = src.crs
    out = np.zeros((H_, W_), np.uint8)
    reproject(arr, out, src_transform=src_tr, src_crs=src_crs, dst_transform=tr_win, dst_crs=config.TARGET_CRS,
              resampling=Resampling.nearest, src_nodata=0, dst_nodata=0)
    return out > 0
PRES = {c: window_presence(c) for c in RETAINED}
feat_of = {}
for c in RETAINED:
    m = CUR.set_index("code").loc[c, "merged_into"]
    feat_of.setdefault(m if m else c, []).append(c)
stem_of = {code_of(p): p.stem for p in lc.efg_paths(AB)} | {m: next(p.stem for p in lc.efg_paths(AB) if p.stem.startswith(m + ".")) for m in MERGES if any(c in RETAINED for c in MERGES[m])}
PARENT_T = json.loads((config.y2y_paths("v3.1").records / "efg_targets.json").read_text())["targets"]
rows = []
for feat, codes in feat_of.items():
    pres = np.zeros((H_, W_), bool)
    for c in codes:
        pres |= PRES[c]
    r = dict(feature=stem_of[feat], code=feat, extent_km2=int((pres & ext_mask).sum()))
    for w in WINDOWS:
        r[f"window{w}_km2"] = int((pres & win_masks[w]).sum()); r[f"window{w}_pct"] = 100 * r[f"window{w}_km2"] / int(win_masks[w].sum())
        r[f"target_window{w}"] = config.efg_target(r[f"window{w}_km2"])
    r[f"share_of_window{W_MAIN}_inside_extent"] = r["extent_km2"] / max(r[f"window{W_MAIN}_km2"], 1)
    r["rare_window"] = bool(r[f"window{W_MAIN}_km2"] <= config.EFG_RARE_WINDOW_PCT * int(win_masks[W_MAIN].sum()))
    for w in W_SENS:
        r[f"rare_window{w}"] = bool(r[f"window{w}_km2"] <= config.EFG_RARE_WINDOW_PCT * int(win_masks[w].sum()))
    r["target_on_extent"] = float(CARD.set_index("feature").loc[stem_of[feat], "target_on_extent"])
    r["target_v31"] = r[f"target_window{W_MAIN}"]
    r["parent_target_v31"] = float(PARENT_T.get(stem_of[feat], np.nan))
    rows.append(r)
WF = pd.DataFrame(rows).sort_values(f"window{W_MAIN}_km2").reset_index(drop=True)
WF.to_csv(REC / "efg_window_footprints.csv", index=False)
print("window areas: " + ", ".join(f"+{w} km = {int(win_masks[w].sum()):,} km2" for w in WINDOWS) + f" | Alberta extent {int(ext_mask.sum()):,} km2")
print(WF[["code", "extent_km2", f"window{W_MAIN}_km2", f"window{W_MAIN}_pct", f"share_of_window{W_MAIN}_inside_extent", "rare_window", "target_on_extent", "target_v31", "parent_target_v31"]
         + [f"target_window{w}" for w in W_SENS]].to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print(f"\nrare in the Alberta +{W_MAIN} km window (<= {100*config.EFG_RARE_WINDOW_PCT:g}% of it): {list(WF[WF.rare_window].code) or 'none'}"
      + "".join(f" | +{w} km: {list(WF[WF[f'rare_window{w}']].code) or 'none'}" for w in W_SENS))
print(f"features at the {config.EFG_TARGET_ANCHORS['floor_t']:.2f} floor: {int((WF.target_v31 <= config.EFG_TARGET_ANCHORS['floor_t'] + 1e-9).sum())}/{len(WF)} "
      f"(parent: {int((WF.parent_target_v31 <= config.EFG_TARGET_ANCHORS['floor_t'] + 1e-9).sum())}/{len(WF)} of the same features)")
T31 = dict(zip(WF.feature, WF.target_v31.round(4)))
(REC / "efg_targets.json").write_text(json.dumps(dict(rule="loglinear", window="alberta_extent", window_km=W_MAIN, sensitivity_km=list(W_SENS),
    anchors=config.EFG_TARGET_ANCHORS, decision="D-AB12 (AB spec v0.5): the parent's v0.17.3 window rule applied to the Alberta extent",
    formula="t from the class footprint within the ALBERTA study extent buffered by window_km (zonal count on the GET archive maps, 1 km nearest warp): "
            "1 for <= full_km2, floor_t for >= floor_km2, log-linear between",
    targets=T31, parent_targets_same_features={k: PARENT_T.get(k) for k in T31}), indent=1))


window areas: +100 km = 346,378 km2, +250 km = 808,652 km2, +500 km = 1,888,012 km2 | Alberta extent 109,225 km2
       code  extent_km2  window250_km2  window250_pct  share_of_window250_inside_extent  rare_window  target_on_extent  target_v31  parent_target_v31  target_window100  target_window500
       T6.1        1062           5597          0.692                             0.190         True             1.000       0.719              0.359             0.729             0.525
       T6.3       15226          34859          4.311                             0.437        False             0.574       0.421              0.100             0.442             0.251
       T6.2       11928          57137          7.066                             0.209        False             0.621       0.341              0.100             0.359             0.200
       T4.4        3924          70004          8.657                             0.056        False             0.949       0.307             

1368

In [4]:
# ---- manifest v3.1: the 12 design formulations, weights re-derived on the AB stack (must equal v1), EFG targets added ---
MODULES = ["config.py", "leverage_core.py", "ensemble_core.py", "prioritizr_core.R", "mga_core.R"]
git = lambda *a: subprocess.run(["git", *a], cwd=ROOT, capture_output=True, text=True)
GIT_SHA = git("rev-parse", "HEAD").stdout.strip()
dirty = [m for m in MODULES if git("diff", "--quiet", "HEAD", "--", m).returncode != 0]
assert not dirty, f"pipeline module(s) modified vs HEAD -- commit before freezing: {dirty}"
MOD_SHA = {m: hashlib.sha256((ROOT / m).read_bytes()).hexdigest() for m in MODULES}
V1P = config.ab_paths("v1")
sha_v1 = hashlib.sha256(V1P.manifest.read_bytes()).hexdigest()
assert sha_v1 == V1P.freeze.read_text().split()[0], "v1 manifest no longer matches its freeze hash -- stop"
V1 = pd.read_csv(V1P.manifest).set_index("formulation_id")
DESIGN = [f for f in V1.index if not V1.loc[f, "scenario_id"].endswith("x")]
assert len(DESIGN) == 12, DESIGN
SC = json.loads((SPEC / "scenarios_ab_v1.json").read_text()); LV = json.loads((SPEC / "ab_budget_levels_v1.json").read_text())
CONSTS = json.loads((HERE / "audit" / "audit_objects_ab" / "audit_constants.json").read_text())
REALIZATION = {"ssp585_2071_2100": None, "ssp245_2071_2100": AB / "climate_realizations" / "macrorefugia_245_2071_2100.tif"}
sha245 = hashlib.sha256(REALIZATION["ssp245_2071_2100"].read_bytes()).hexdigest()
def _norm(d, what):
    tot = sum(d.values()); assert abs(tot - 1.0) < 5e-3, f"{what} sums to {tot:.6f}"
    return {k: v / tot for k, v in d.items()}
S0 = SC["S0_balanced"]; recipes = {}
for name in ("S0_balanced", "S1_core_habitat", "S2_connectivity", "S3_biodiversity", "S4_carbon"):
    s = SC[name]
    recipes[name.split("_")[0].lower()] = dict(shares=_norm(s["block_shares"], name), within={b: _norm(m, f"{name}/{b}") for b, m in s["within_block"].items()}, targets=s["targets"], extra={})
recipes["s5"] = dict(shares=_norm(S0["block_shares"], "S5"), within={b: _norm(m, f"S5/{b}") for b, m in S0["within_block"].items()}, targets=S0["targets"], extra={"human_modification": 10.0})
efg_hashes = {p.stem: hashlib.sha256(p.read_bytes()).hexdigest() for p in lc.efg_paths(AB)}
efg_block_sha = hashlib.sha256("\n".join(f"{k}:{v}" for k, v in sorted(efg_hashes.items())).encode()).hexdigest()
now = datetime.now(timezone.utc).isoformat(); rows = []
for fid in DESIGN:
    v1 = V1.loc[fid]; r = recipes[v1.scenario_id]; climate = v1.climate_level
    lp = {"climate_type_macrorefugia": REALIZATION[climate]} if REALIZATION[climate] is not None else None
    d = lc.scenario_weights(r["shares"], within_block=r["within"], targets=r["targets"], handoff_dir=AB, layer_paths=lp)
    w = {x.feature: round(x.w, 6) for x in d.itertuples()}; w.update(r["extra"])
    w1 = json.loads(v1.weight_vector)
    assert set(w) == set(w1) and all(abs(w[k] - w1[k]) < 1e-6 for k in w), f"{fid}: weights differ from v1 (EFGs are outside the block accounting)"
    hashes = json.loads(v1.input_layer_hashes); hashes = {k: v for k, v in hashes.items() if k not in efg_hashes}; hashes.update(efg_hashes)
    row = v1.to_dict(); row["formulation_id"] = fid
    row.update(target_vector=json.dumps({**json.loads(v1.target_vector), **T31}), mirror_spec_version="v0.5", role="design",
               anchor_ref="", twin_ref="", mga_ref="",                                   # everything re-solved on the curated block
               applied_band_g=0.02, applied_band_rule="D-AB13: guarded g = 5% (mirror) unless the 5% frequent tier is flat (< 100 km2), then g = 2% (D-AB10); decided in 11",
               input_layer_hashes=json.dumps(hashes), manifest_version=3.1, efg_block_version="v3", efg_block_sha256=efg_block_sha,
               efg_features=json.dumps(sorted(efg_hashes)), efg_target_rule=f"loglinear_window{W_MAIN}km_alberta_extent",
               supersedes=f"v1 {sha_v1[:16]}", trigger="AB spec v0.5 (parent study plan v0.17-v0.17.3; R7.9-R7.11 / R10.18): R0 curation 27 -> 15 classes / 13 features + window-derived targets",
               pipeline_git_sha=GIT_SHA, pipeline_module_sha256=json.dumps(MOD_SHA), created_utc=now, frozen=True)
    rows.append(row)
M31 = pd.DataFrame(rows); assert M31.formulation_id.is_unique and len(M31) == 12
if VP.manifest.exists() and VP.freeze.exists() and hashlib.sha256(VP.manifest.read_bytes()).hexdigest() == VP.freeze.read_text().split()[0]:
    print(f"manifest v3.1 already frozen ({VP.freeze.read_text().split()[0][:16]}...) -- kept byte-identical (supersede, never delete)")
else:
    M31.to_csv(VP.manifest, index=False)
    d31 = hashlib.sha256(VP.manifest.read_bytes()).hexdigest(); VP.freeze.write_text(f"{d31}  {VP.manifest.name}\n")
    print(f"FROZEN: {VP.manifest.relative_to(ROOT)} (12 design formulations at level {M31.budget_level.iloc[0]}; EFG block v3 sha {efg_block_sha[:16]}...; "
          f"targets from the Alberta +{W_MAIN} km window) | sha256 {d31[:16]}...")
print("EFG targets in every target_vector: " + ", ".join(f"{code_of(pathlib.Path(k))} {v:.2f}" for k, v in sorted(T31.items(), key=lambda kv: kv[1])))
print("commit spec/manifest_v3.1.csv + spec/manifest_v3.1.sha256 + spec/v3.1/ -- that commit is the pre-registration of the re-solve; then 10_ab4_ensemble (R)")


FROZEN: analyses/alberta_prioritization/spec/manifest_v3.1.csv (12 design formulations at level A; EFG block v3 sha c8ee930b64e00bca...; targets from the Alberta +250 km window) | sha256 6e66ec3f9c9517ad...
EFG targets in every target_vector: T6.4 0.10, F2.4 0.10, T2.1 0.10, T2.2 0.16, F1.3 0.18, T5.1 0.19, SF1.2 0.20, TF1.6_TF1.7 0.21, S1.1_SF1.1 0.29, T4.4 0.31, T6.2 0.34, T6.3 0.42, T6.1 0.72
commit spec/manifest_v3.1.csv + spec/manifest_v3.1.sha256 + spec/v3.1/ -- that commit is the pre-registration of the re-solve; then 10_ab4_ensemble (R)


## → next

Commit the freeze, then `10_ab4_ensemble` (R; VERSION v3.1, anchors + twins + MGA at both bands, no k-best) → `11_ab4_analysis`
(decides the applied band by rule D-AB13, M4.31 disclosure) → `11b_ab_e19_solves` (R) → `11c_ab_e19_analysis` → `12` → `13` —
or one command: `caffeinate -i bash analyses/alberta_prioritization/run_v31.sh`. Log the block card, the window targets and
the freeze in `spec/results_log.md` (R8.1–R8.2).